# DNNLS Live Arena — Two Moons starter notebook

This notebook shows the two ways to produce a submission for the **Two Moons — Decision Boundary Challenge**:

* **Option A – prediction CSV** (safest, no code is executed on the server), and
* **Option B – Python bundle** (`submission.zip` with `submission.py` + `weights.pt`, run in a sandbox).

Workflow: train here in Colab → download the artifact → upload it on the Arena student page in your browser.

> This notebook is an *export helper*, not a model-tuning exercise. Replace the simple models with your own ideas.


## 0. Get the public challenge files

The three public files (`train.csv`, `test_features.csv`, `visualization_grid.csv`) are committed in the
project repository, so Colab can download them straight from GitHub. The Arena server also serves them,
but a server on your lecturer's laptop (classroom Wi-Fi, `localhost`, ...) is *not* reachable from Colab,
which runs in Google's cloud.

Order of attempts: GitHub → Arena URL → manual upload with the file picker.


In [ ]:
DATA_BASE_URL = "https://raw.githubusercontent.com/aljiro/interactive_coding/main/examples/data"   # public challenge files in the repository
ARENA_URL = "http://localhost:8000"  # address shown on the projector (only reachable from Colab if deployed on a public VM)
CHALLENGE = "moons"
FILES = ["train.csv", "test_features.csv", "visualization_grid.csv"]

import os, requests

def fetch(base_url):
    for name in FILES:
        r = requests.get(f"{base_url}/{name}", timeout=10)
        r.raise_for_status()
        if not r.text.startswith("id,"):
            raise ValueError(f"{name}: unexpected content")
        open(name, "wb").write(r.content)
        print("downloaded", name, len(r.content), "bytes")

try:
    fetch(DATA_BASE_URL)
except Exception as exc:
    print("GitHub download failed:", exc)
    try:
        fetch(f"{ARENA_URL}/api/challenges/{CHALLENGE}/resources")
    except Exception as exc2:
        print("Arena download failed:", exc2)
        try:
            from google.colab import files  # type: ignore
            print("\nUpload train.csv, test_features.csv and visualization_grid.csv (download them from the Arena student page):")
            files.upload()
        except ImportError:
            pass

assert all(os.path.exists(f) for f in FILES), "missing challenge files"


## 1. Load the data

In [ ]:
import numpy as np
import pandas as pd

train = pd.read_csv("train.csv")
test_features = pd.read_csv("test_features.csv")          # id, x1, x2  (labels are hidden on the server)
grid = pd.read_csv("visualization_grid.csv")              # id, x1, x2  (100 x 100 points)

X_train = train[["x1", "x2"]].to_numpy(dtype=np.float32)
y_train = train["label"].to_numpy(dtype=np.int64)
X_test = test_features[["x1", "x2"]].to_numpy(dtype=np.float32)
X_grid = grid[["x1", "x2"]].to_numpy(dtype=np.float32)

print(X_train.shape, X_test.shape, X_grid.shape, "class balance:", y_train.mean().round(3))


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 3.4))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", s=12, edgecolor="none")
plt.xlabel("x1"); plt.ylabel("x2"); plt.title("Two moons – training data"); plt.gca().set_aspect("equal")
plt.show()


## 2. Option A — train a classifier and export a prediction CSV

The CSV must contain **one row for every `test` id and every `grid` id**:

```
dataset,id,probability
test,0,0.021
...
grid,0,0.001
...
```

`probability` is the predicted probability of class 1 (a finite number in [0, 1]).


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=0)
clf = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(32, 32), max_iter=2000, random_state=0))
clf.fit(X_tr, y_tr)
print("validation accuracy:", round(clf.score(X_val, y_val), 3))

clf.fit(X_train, y_train)   # refit on everything before predicting the hidden test set


In [ ]:
def write_prediction_csv(path, p_test, p_grid):
    rows = pd.concat([
        pd.DataFrame({"dataset": "test", "id": test_features["id"], "probability": p_test}),
        pd.DataFrame({"dataset": "grid", "id": grid["id"], "probability": p_grid}),
    ])
    rows["probability"] = rows["probability"].clip(0, 1).round(5)
    rows.to_csv(path, index=False)
    print(f"wrote {path}: {len(rows)} rows")

p_test = clf.predict_proba(X_test)[:, 1]
p_grid = clf.predict_proba(X_grid)[:, 1]
write_prediction_csv("predictions.csv", p_test, p_grid)
pd.read_csv("predictions.csv").head()


In [ ]:
# Download it, then upload predictions.csv on the Arena student page.
try:
    from google.colab import files  # type: ignore
    files.download("predictions.csv")
except ImportError:
    print("Not in Colab: predictions.csv is in the current directory.")


## 3. Option B — a PyTorch model exported as a Python bundle

The server unpacks `submission.zip` in an isolated container (no network, 1 CPU, 512 MB, 30 s) and calls

```python
def predict(X):   # X: numpy (N, 2)  ->  numpy (N,) probabilities of class 1
```

Everything your `predict` needs (architecture, weight loading) must live in `submission.py` and load files **relative to `__file__`**.
We write `submission.py` first so the notebook and the bundle share one model definition.


In [ ]:
%%writefile submission.py
import os
import numpy as np
import torch
import torch.nn as nn

HERE = os.path.dirname(os.path.abspath(__file__))


class MLP(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


_model = None


def _load():
    global _model
    if _model is None:
        m = MLP()
        m.load_state_dict(torch.load(os.path.join(HERE, "weights.pt"), map_location="cpu"))
        m.eval()
        _model = m
    return _model


def predict(X):
    """X: numpy.ndarray (N, 2) -> numpy.ndarray (N,) with P(class 1)."""
    model = _load()
    with torch.no_grad():
        logits = model(torch.as_tensor(np.asarray(X, dtype=np.float32)))
        return torch.sigmoid(logits).numpy().astype(np.float64)


In [ ]:
import importlib, torch, torch.nn.functional as F
import submission
importlib.reload(submission)

torch.manual_seed(0)
model = submission.MLP()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
Xt = torch.as_tensor(X_train); yt = torch.as_tensor(y_train, dtype=torch.float32)
for epoch in range(400):
    opt.zero_grad()
    loss = F.binary_cross_entropy_with_logits(model(Xt), yt)
    loss.backward(); opt.step()
    if epoch % 100 == 0:
        acc = ((model(Xt) > 0).float() == yt).float().mean().item()
        print(f"epoch {epoch:3d}  loss {loss.item():.3f}  train acc {acc:.3f}")

torch.save(model.state_dict(), "weights.pt")
print("saved weights.pt")


In [ ]:
# Sanity-check the bundle exactly as the server will use it, then zip it.
import zipfile
importlib.reload(submission)
p = submission.predict(X_test[:5])
print("predict() ->", p.shape, p.round(3))
assert p.shape == (5,) and np.all((p >= 0) & (p <= 1))

with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("submission.py")
    zf.write("weights.pt")
print("submission.zip:", [i.filename for i in zipfile.ZipFile("submission.zip").infolist()])

try:
    from google.colab import files  # type: ignore
    files.download("submission.zip")
except ImportError:
    print("Not in Colab: submission.zip is in the current directory.")


## 4. (Optional) submit directly from Colab

Only works when the Arena URL is reachable from Colab (e.g. deployed on a university VM). Paste your participant token from the browser's local storage (`dnnls.participant`) or join via the API below.


In [ ]:
SESSION_CODE = ""      # join code shown on the projector
DISPLAY_NAME = ""      # your name on the projector
ARTIFACT = "predictions.csv"   # or "submission.zip"

if SESSION_CODE and DISPLAY_NAME:
    r = requests.post(f"{ARENA_URL}/api/sessions/join", json={"join_code": SESSION_CODE, "display_name": DISPLAY_NAME}, timeout=10)
    r.raise_for_status()
    joined = r.json()
    with open(ARTIFACT, "rb") as fh:
        r = requests.post(f"{ARENA_URL}/api/sessions/{joined['session']['id']}/submissions",
                          headers={"X-Participant-Token": joined["token"]}, files={"file": (ARTIFACT, fh)}, timeout=60)
    print(r.status_code, r.json())
